# Semáforo de salud financiera empresarial

**Proyecto integrador** — Inteligencia de Negocios · Análisis de Datos · Aprendizaje Computacional
Instituto Tecnológico Metropolitano (ITM) · Equipo: Julián, Santiago, Jacobo y Oscar

## 0. Problema de negocio

Cuando una empresa entra en dificultades financieras, el problema suele verse en sus estados
financieros antes de que se vuelva una crisis: la liquidez cae, el endeudamiento sube o el
patrimonio se vuelve negativo. Detectar esas señales a tiempo le sirve a varios actores:

- **Entes de supervisión** (como la Superintendencia de Sociedades), para priorizar a qué
  empresas hacerles seguimiento.
- **Bancos y proveedores**, para decidir a quién otorgar crédito o plazos de pago.
- **Las mismas empresas**, para compararse con su sector y corregir el rumbo.

**Pregunta de negocio:** con el balance de una empresa en el año *t*, ¿podemos anticipar si
estará en **deterioro financiero** en el año *t+1*?

**Fuentes de datos** (públicas, Portal de Datos Abiertos de Colombia):

1. *Estados Financieros NIIF – Estado de Situación Financiera* (Supersociedades): balances de las
   empresas, una fila por empresa, cuenta y fecha de corte.
2. *10.000 Empresas más Grandes del País* (Supersociedades): sector, región, ciudad, ingresos y
   ganancia de cada empresa.

**Estructura del notebook:**

| Sección | Etapa |
|---|---|
| 1 | ETL: extracción, limpieza, integración y modelo estrella |
| 2 | Análisis exploratorio e inferencia estadística |
| 3 | Preparación de datos y modelos de aprendizaje supervisado |
| 4 | Conclusiones, recomendaciones y consideraciones éticas |

El dashboard de Power BI está en la carpeta `dashboard/`.

## 1. ETL — Extracción, transformación y carga

### 1.1 Extracción de datos por API

Las dos bases se publican en el Portal de Datos Abiertos, que usa la plataforma **Socrata**.
Socrata permite consultarlas con un lenguaje parecido a SQL (SoQL) directamente en la URL:

- `$select`: columnas a traer.
- `$where`: filtros.
- `$limit` y `$offset`: paginación.

Descargar por API, en lugar de bajar un archivo a mano, deja el proceso **automatizado y
reproducible**: cualquiera del equipo puede volver a ejecutar esta sección y obtener los mismos
datos.

La base de balances tiene cerca de **19,6 millones de filas**, así que no se descarga completa.
Se filtra en el servidor para traer solo lo que el proyecto necesita.

#### Configuración

Las rutas se construyen de forma **relativa** a la raíz del repositorio. Así el notebook
funciona en el computador de cualquier integrante.

In [1]:
import os
import io
import time
from pathlib import Path

import pandas as pd
import requests

# Raíz del repositorio: si el notebook se abre desde /notebooks, se sube un nivel
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_RAW = RAIZ / "data" / "raw"
DATA_PROCESSED = RAIZ / "data" / "processed"
DATA_RAW.mkdir(parents=True, exist_ok=True)
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

# Identificadores de las bases en datos.gov.co
URL_BASE = "https://www.datos.gov.co/resource/{}.csv"
ID_BALANCES = "pfdp-zks5"   # Estados Financieros NIIF - Estado de Situación Financiera
ID_EMPRESAS = "6cat-2gcs"   # 10.000 Empresas más Grandes del País

# Años de corte a descargar (balances de cierre de año)
ANIOS = list(range(2019, 2026))

# Token opcional de datos.gov.co (variable de entorno, nunca escrito en el código)
APP_TOKEN = os.getenv("DATOS_GOV_TOKEN")

print("Raíz del proyecto:", RAIZ.name)
print("Token configurado:", "sí" if APP_TOKEN else "no (la descarga funciona, pero puede ser más lenta)")

Raíz del proyecto: Final-AnalisisDatos
Token configurado: no (la descarga funciona, pero puede ser más lenta)


#### Función de descarga paginada

La API devuelve como máximo un número fijo de filas por consulta. La función pide los datos por
páginas de 50.000 filas hasta que una página llega vacía.

- El parámetro `$order=:id` ordena por el identificador interno de cada fila. Así ninguna fila
  se repite ni se pierde entre páginas.
- Si una página falla por un error de red, se reintenta hasta 3 veces antes de detenerse.
- Todo se lee como texto (`dtype=str`). La conversión de tipos se hace en la etapa de limpieza.

In [2]:
def descargar_socrata(id_base, params, tam_pagina=50_000, reintentos=3):
    """Descarga una consulta SoQL completa, página por página, y la devuelve como DataFrame."""
    url = URL_BASE.format(id_base)
    encabezados = {"X-App-Token": APP_TOKEN} if APP_TOKEN else {}
    paginas, offset = [], 0

    while True:
        # Orden por defecto: identificador interno de la fila (se puede cambiar con params)
        consulta = {"$order": ":id", **params, "$limit": tam_pagina, "$offset": offset}
        for intento in range(1, reintentos + 1):
            try:
                resp = requests.get(url, params=consulta, headers=encabezados, timeout=300)
                resp.raise_for_status()
                break
            except requests.RequestException as error:
                if intento == reintentos:
                    raise
                print(f"  Fallo de conexión en la página (offset={offset}), "
                      f"reintento {intento} de {reintentos - 1}...")
                time.sleep(5 * intento)

        if not resp.text.strip():
            break
        pagina = pd.read_csv(io.StringIO(resp.text), dtype=str)
        if pagina.empty:
            break
        paginas.append(pagina)
        offset += tam_pagina
        print(f"  {offset - tam_pagina + len(pagina):,} filas descargadas")
        if len(pagina) < tam_pagina:
            break

    return pd.concat(paginas, ignore_index=True) if paginas else pd.DataFrame()

#### Base 2: 10.000 Empresas más Grandes del País

Esta base se descarga primero y completa, porque es pequeña (50.000 filas: 10.000 empresas por
cada año de corte entre 2021 y 2025). Define el **universo de análisis**: solo se trabajará con
las empresas que aparecen en ella, que son las que tienen información de sector y ubicación.

Una revisión previa de la API mostró que el NIT viene como texto con decimales (por ejemplo
`"899999068.00"`). Aquí solo se le quita ese `.00` para poder filtrar la otra base. La
normalización completa del NIT se hace en la etapa de integración.

In [3]:
empresas_raw = descargar_socrata(ID_EMPRESAS, params={})
empresas_raw.to_csv(DATA_RAW / "empresas_10000_raw.csv", index=False)

print("Dimensiones:", empresas_raw.shape)
print("Columnas:", list(empresas_raw.columns))
empresas_raw.head()

  50,000 filas descargadas
Dimensiones: (50000, 14)
Columnas: ['nit', 'raz_n_social', 'supervisor', 'regi_n', 'departamento_domicilio', 'ciudad_domicilio', 'ciiu', 'macrosector', 'ingresos_operacionales', 'ganancia_p_rdida', 'total_activos', 'total_pasivos', 'total_patrimonio', 'a_o_de_corte']


,nit,raz_n_social,supervisor,regi_n,departamento_domicilio,ciudad_domicilio,ciiu,macrosector,ingresos_operacionales,ganancia_p_rdida,total_activos,total_pasivos,total_patrimonio,a_o_de_corte
0,899999068.00,ECOPETROL S.A.,SUPERFINANCIERA,BOGOTÁ - CUNDINAMARCA,BOGOTA D.C.,"BOGOTA, D.C.",610,MINERO,$ 100.59,$ 9.03,$ 198.11,$ 114.35,$ 83.76,2025.00
1,830095213.00,ORGANIZACIÓN TERPEL S.A.,SUPERFINANCIERA,BOGOTÁ - CUNDINAMARCA,BOGOTA D.C.,"BOGOTA, D.C.",4661,COMERCIO,$ 26.40,$ 0.63,$ 8.11,$ 4.70,$ 3.41,2025.00
2,900112515.00,REFINERIA DE CARTAGENA S.A.,SUPERSERVICIOS,REGIÓN CARIBE,BOLIVAR,CARTAGENA,1921,MANUFACTURA,$ 22.40,-$ 0.82,$ 34.95,$ 11.57,$ 23.38,2025.00
3,900276962.00,D1 S.A.S.,SUPERSOCIEDADES,BOGOTÁ - CUNDINAMARCA,BOGOTA D.C.,"BOGOTA, D.C.",4711,COMERCIO,$ 21.61,$ 0.42,$ 6.94,$ 6.72,$ 0.21,2025.00
4,890904996.00,EMPRESAS PÚBLICAS DE MEDELLÍN E.S.P.,SUPERFINANCIERA,ANTIOQUIA,ANTIOQUIA,MEDELLÍN,3513,SERVICIOS,$ 20.29,$ 4.88,$ 71.08,$ 36.11,$ 34.97,2025.00


In [4]:
# NIT sin decimales, solo para filtrar los balances
empresas_raw["nit_filtro"] = empresas_raw["nit"].str.strip().str.replace(r"\.0+$", "", regex=True)
NITS_UNIVERSO = set(empresas_raw["nit_filtro"].dropna())

print("Filas por año de corte:")
print(empresas_raw["a_o_de_corte"].value_counts().sort_index())
print(f"\nEmpresas distintas en el universo: {len(NITS_UNIVERSO):,}")

Filas por año de corte:
a_o_de_corte
2021.00    10000
2022.00    10000
2023.00    10000
2024.00    10000
2025.00    10000
Name: count, dtype: int64

Empresas distintas en el universo: 15,032


> **Interpretación:** la base tiene 50.000 registros: 10.000 empresas por año de corte entre 2021 y 2025. Como muchas empresas aparecen en varios años, el universo queda en **15.032 empresas distintas**. Todas las columnas llegan como texto: el NIT y el año de corte traen decimales (`899999068.00`, `2025.00`) y los montos traen el signo `$` y están expresados en billones de pesos, así que se convierten en la etapa de limpieza. El 87,5 % de los registros corresponde a empresas vigiladas por Supersociedades; el resto lo vigilan otras superintendencias (Supersalud, Supertransporte, Supervigilancia, Superservicios y Superfinanciera).

#### Inventario de cuentas disponibles

En la base de balances, cada cuenta contable aparece en la columna `concepto` con su nombre
textual. Antes de descargar hay que saber **cómo se llaman exactamente** las cuentas que
necesitamos.

Para no consultar los 19,6 millones de filas, el inventario se hace solo sobre el corte del
31 de diciembre de 2024. El resultado se guarda en `data/raw/conceptos_disponibles.csv` como
evidencia del proceso.

Filtros que se aplican, tanto aquí como en la descarga:

- `periodo = 'Periodo Actual'`: cada reporte también trae el año anterior como comparativo.
  Ese dato ya está en el reporte del año anterior, así que se descarta para no duplicarlo.
- `punto_entrada` sin las palabras *Consolidado* ni *Combinado*: se usan los estados
  **individuales** y **separados** de cada empresa. Los consolidados y combinados suman varias
  entidades (por ejemplo, una empresa y sus subordinadas), así que no son comparables con los
  demás.

In [5]:
FILTRO_COMUN = ("periodo = 'Periodo Actual' AND punto_entrada not like '%Consolidado%' "
                "AND punto_entrada not like '%Combinado%'")

conceptos = descargar_socrata(
    ID_BALANCES,
    params={
        "$select": "concepto, count(*) AS n",
        "$where": f"fecha_corte = '2024-12-31T00:00:00.000' AND {FILTRO_COMUN}",
        "$group": "concepto",
        "$order": "n DESC",
    },
)
conceptos["n"] = conceptos["n"].astype(int)
conceptos = conceptos.sort_values("n", ascending=False).reset_index(drop=True)
conceptos.to_csv(DATA_RAW / "conceptos_disponibles.csv", index=False)

print(f"Cuentas distintas en el corte 2024: {len(conceptos):,}")
conceptos.head(60)

  71 filas descargadas
Cuentas distintas en el corte 2024: 71


,concepto,n
0,Total de activos,29427
1,Patrimonio total,29427
2,Efectivo y equivalentes al efectivo,29427
3,Total pasivos,29427
4,Total de patrimonio y pasivos,29427
5,Total activos corrientes distintos de los acti...,29427
6,Activos corrientes totales,29427
7,Capital emitido,29426
8,Ganancias acumuladas,29408
9,Pasivos corrientes totales,29221


#### Selección de cuentas

Para calcular los indicadores financieros del proyecto se necesitan las 10 cuentas de la tabla
siguiente. Una misma cuenta puede aparecer con nombres distintos según la taxonomía (NIIF Plenas
o NIIF Pymes), por eso cada cuenta tiene una **lista** de nombres posibles.

| Cuenta | Para qué se usa |
|---|---|
| Activo corriente | Liquidez, prueba ácida, capital de trabajo |
| Activo total | Endeudamiento, solvencia |
| Pasivo corriente | Liquidez, prueba ácida, capital de trabajo |
| Pasivo total | Endeudamiento, validación contable |
| Patrimonio total | Solvencia, variable objetivo, validación contable |
| Efectivo y equivalentes | Liquidez inmediata |
| Inventarios | Prueba ácida |
| Deudores comerciales | Calidad del activo corriente |
| Capital emitido | Contexto patrimonial |
| Ganancias acumuladas | Historial de resultados |

La celda que sigue a la de las listas verifica qué nombres existen realmente en el inventario.
**Si alguna cuenta sale sin coincidencias**, se busca su nombre en
`conceptos_disponibles.csv` y se agrega a la lista correspondiente antes de descargar.

In [6]:
# Nombres según la taxonomía NIIF en español. "Patrimonio total" y "Capital emitido" se
# verificaron en la API; los demás se confirman con la celda siguiente.
CUENTAS = {
    "activo_corriente":     ["Activos corrientes totales"],
    "activo_total":         ["Total de activos"],
    "pasivo_corriente":     ["Pasivos corrientes totales"],
    "pasivo_total":         ["Total pasivos"],
    "patrimonio_total":     ["Patrimonio total"],
    "efectivo":             ["Efectivo y equivalentes al efectivo"],
    "inventarios":          ["Inventarios corrientes", "Inventarios"],
    "deudores_comerciales": ["Cuentas comerciales por cobrar y otras cuentas por cobrar corrientes"],
    "capital_emitido":      ["Capital emitido"],
    "ganancias_acumuladas": ["Ganancias acumuladas"],
}

In [7]:
disponibles = set(conceptos["concepto"])
faltantes = []
for cuenta, nombres in CUENTAS.items():
    encontrados = [n for n in nombres if n in disponibles]
    estado = "OK" if encontrados else "SIN COINCIDENCIA"
    print(f"{cuenta:22s} {estado:17s} {encontrados}")
    if not encontrados:
        faltantes.append(cuenta)

# Ayuda para corregir: nombres del inventario que contienen una palabra clave de cada faltante
PALABRAS_CLAVE = {
    "activo_corriente": "activos corrientes", "activo_total": "activos",
    "pasivo_corriente": "pasivos corrientes", "pasivo_total": "pasivos",
    "patrimonio_total": "patrimonio", "efectivo": "efectivo", "inventarios": "inventario",
    "deudores_comerciales": "por cobrar", "capital_emitido": "capital",
    "ganancias_acumuladas": "ganancias acumuladas",
}
for cuenta in faltantes:
    clave = PALABRAS_CLAVE[cuenta]
    sugeridos = conceptos[conceptos["concepto"].str.lower().str.contains(clave, na=False)]
    print(f"\nPosibles nombres para '{cuenta}':")
    print(sugeridos.head(10).to_string(index=False))

activo_corriente       OK                ['Activos corrientes totales']
activo_total           OK                ['Total de activos']
pasivo_corriente       OK                ['Pasivos corrientes totales']
pasivo_total           OK                ['Total pasivos']
patrimonio_total       OK                ['Patrimonio total']
efectivo               OK                ['Efectivo y equivalentes al efectivo']
inventarios            OK                ['Inventarios corrientes']
deudores_comerciales   OK                ['Cuentas comerciales por cobrar y otras cuentas por cobrar corrientes']
capital_emitido        OK                ['Capital emitido']
ganancias_acumuladas   OK                ['Ganancias acumuladas']


> **Interpretación:** las 10 cuentas necesarias aparecen con el mismo nombre en el corte de 2024, así que no hubo que ajustar el diccionario `CUENTAS`. En ese corte hay 71 cuentas distintas y las principales (activos, pasivos, patrimonio, efectivo, capital) las reportan unas 29.400 empresas. La excepción es inventarios corrientes, que aparece en 18.795 reportes (64 %), porque muchas empresas de servicios no manejan inventario.

#### Base 1: balances por año

La descarga se hace **un año a la vez**. Así cada consulta es más liviana y se evitan los tiempos
de espera agotados en el servidor. Después de cada año se conservan solo las empresas del
universo definido por la base 2, lo que reduce mucho el tamaño del archivo final.

Por cada fila se descargan:

- `nit`, `fecha_corte` y `concepto`.
- `valor`: sigue como texto; se convierte en la limpieza.
- `punto_entrada` y `taxonomia`: indican el tipo de estado financiero.
- `numero_radicado` y `codigo_instancia`: sirven en la limpieza para detectar
  retransmisiones del mismo reporte.

In [8]:
nombres_cuentas = sorted({n for lista in CUENTAS.values() for n in lista})
lista_soql = ", ".join("'" + n.replace("'", "''") + "'" for n in nombres_cuentas)
columnas = "nit, fecha_corte, punto_entrada, taxonomia, numero_radicado, codigo_instancia, concepto, valor"

balances_por_anio = []
for anio in ANIOS:
    print(f"Año {anio}:")
    df_anio = descargar_socrata(
        ID_BALANCES,
        params={
            "$select": columnas,
            "$where": (f"fecha_corte = '{anio}-12-31T00:00:00.000' AND {FILTRO_COMUN} "
                       f"AND concepto in ({lista_soql})"),
        },
    )
    if df_anio.empty:
        print("  sin datos")
        continue
    df_anio = df_anio[df_anio["nit"].str.strip().isin(NITS_UNIVERSO)]
    print(f"  {len(df_anio):,} filas de empresas del universo")
    balances_por_anio.append(df_anio)

balances_raw = pd.concat(balances_por_anio, ignore_index=True)
balances_raw.to_csv(DATA_RAW / "balances_raw.csv", index=False)
print("\nDimensiones finales:", balances_raw.shape)

Año 2019:
  50,000 filas descargadas
  100,000 filas descargadas
  150,000 filas descargadas
  200,000 filas descargadas
  246,654 filas descargadas
  93,378 filas de empresas del universo
Año 2020:
  Fallo de conexión en la página (offset=0), reintento 1 de 2...
  50,000 filas descargadas
  100,000 filas descargadas
  150,000 filas descargadas
  200,000 filas descargadas
  250,000 filas descargadas
  279,152 filas descargadas
  98,023 filas de empresas del universo
Año 2021:
  50,000 filas descargadas
  100,000 filas descargadas
  150,000 filas descargadas
  200,000 filas descargadas
  250,000 filas descargadas
  280,975 filas descargadas
  103,061 filas de empresas del universo
Año 2022:
  50,000 filas descargadas
  100,000 filas descargadas
  150,000 filas descargadas
  200,000 filas descargadas
  250,000 filas descargadas
  300,000 filas descargadas
  350,000 filas descargadas
  400,000 filas descargadas
  450,000 filas descargadas
  500,000 filas descargadas
  550,000 filas descar

#### Resumen de la extracción

Se verifica qué se descargó: filas por año, empresas distintas por año y filas por cuenta. Si
algún año o alguna cuenta trae muy pocos datos, hay que revisar los filtros antes de pasar a la
limpieza.

In [9]:
balances_raw["anio"] = balances_raw["fecha_corte"].str[:4]

resumen = balances_raw.groupby("anio").agg(
    filas=("valor", "size"),
    empresas=("nit", "nunique"),
)
print(resumen, "\n")
print("Filas por cuenta:")
print(balances_raw["concepto"].value_counts())
print("\nTipos de estado financiero (punto de entrada):")
print(balances_raw["punto_entrada"].value_counts())
print("Empresas distintas con balance:", balances_raw["nit"].nunique())

       filas  empresas
anio                  
2019   93378      9495
2020   98023      9950
2021  103061     10435
2022  212064     10990
2023  114737     11647
2024  113098     11490
2025  115996     11822 

Filas por cuenta:
concepto
Total de activos                                                        86657
Total pasivos                                                           86657
Activos corrientes totales                                              86657
Efectivo y equivalentes al efectivo                                     86657
Patrimonio total                                                        86657
Capital emitido                                                         86653
Ganancias acumuladas                                                    86642
Pasivos corrientes totales                                              86586
Cuentas comerciales por cobrar y otras cuentas por cobrar corrientes    85845
Inventarios corrientes                                        

> **Interpretación:** Se descargaron balances de 12.761 de las 15.032 empresas del universo (84,9%). Las demás están vigiladas por otras superintendencias (Supersalud, Supertransporte, entre otras) y no reportan sus estados financieros a Supersociedades. El número de empresas con balance crece de 9.495 en 2019 a 11.822 en 2025. Las cuentas principales tienen alrededor de 86.600 registros; inventarios tiene menos (71.346) porque muchas empresas de servicios no manejan inventario. 2022 tiene casi el doble de filas que los demás años porque la fuente trae registros duplicados exactos; se eliminan en la etapa de limpieza.

**Archivos generados en `data/raw/`** (no se suben al repositorio; se regeneran ejecutando esta
sección):

| Archivo | Contenido |
|---|---|
| `empresas_10000_raw.csv` | Base 2 completa |
| `conceptos_disponibles.csv` | Inventario de cuentas del corte 2024 |
| `balances_raw.csv` | Balances 2019–2025 de las empresas del universo, formato largo |

*Siguiente paso:* limpieza y transformación de los balances.

### 1.2 Limpieza y transformación de los balances

La base de balances llega en **formato largo** (una fila por empresa, cuenta y fecha) y con todos
sus campos como texto. Para poder calcular indicadores hay que dejarla en una tabla con **una fila
por empresa y año** y una columna numérica por cuenta.

Pasos de esta sección:

1. Convertir los tipos de dato: `valor` a número y `fecha_corte` a fecha.
2. Eliminar filas duplicadas exactas.
3. Eliminar retransmisiones: si una empresa envió varios reportes para el mismo corte, quedarse
   con el más reciente.
4. Homologar los nombres de las cuentas entre taxonomías con un diccionario.
5. Pivotear de formato largo a ancho y guardar el resultado en `data/processed/`.

La sección parte del archivo `data/raw/balances_raw.csv` que dejó la extracción, así que no
vuelve a consultar la API. Se lleva una **trazabilidad** de cuántas filas se eliminan en cada
paso y por qué.

In [10]:
balances = pd.read_csv(DATA_RAW / "balances_raw.csv", dtype=str)

# Trazabilidad: (paso, filas que quedan, motivo)
trazabilidad = [("Carga de balances_raw.csv", len(balances), "Archivo generado en la extracción")]

print("Dimensiones:", balances.shape)
print("Tipos de dato:")
print(balances.dtypes)
balances.head()

Dimensiones: (850357, 8)
Tipos de dato:
nit                 str
fecha_corte         str
punto_entrada       str
taxonomia           str
numero_radicado     str
codigo_instancia    str
concepto            str
valor               str
dtype: object


,nit,fecha_corte,punto_entrada,taxonomia,numero_radicado,codigo_instancia,concepto,valor
0,830130401,2019-12-31T00:00:00.000,40 NIIF Pymes - Individuales,01- Estados Financieros de Fin de Ejercicio - ...,2020-01-567458,216651,Total de activos,2761152
1,860029445,2019-12-31T00:00:00.000,40 NIIF Pymes - Individuales,01- Estados Financieros de Fin de Ejercicio - ...,2020-01-251223,169995,Capital emitido,2000000
2,890206034,2019-12-31T00:00:00.000,40 NIIF Pymes - Individuales,01- Estados Financieros de Fin de Ejercicio - ...,2020-01-209387,159193,Cuentas comerciales por cobrar y otras cuentas...,30135944
3,811002545,2019-12-31T00:00:00.000,40 NIIF Pymes - Individuales,01- Estados Financieros de Fin de Ejercicio - ...,2020-01-249683,169879,Capital emitido,100000
4,860051688,2019-12-31T00:00:00.000,10 NIIF Plenas - Individuales,01- Estados Financieros de Fin de Ejercicio - ...,2020-01-149004,153214,Capital emitido,2200000


#### Conversión de tipos de dato

- **`valor`**: viene como texto. Se convierte a número con `pd.to_numeric`; si algún valor no se
  puede convertir, queda como nulo (`errors="coerce"`) y se cuenta para revisarlo.
- **`fecha_corte`**: se convierte a fecha y de ella se saca el **año** del balance.
- **`numero_radicado`**: tiene la forma `AAAA-MM-consecutivo` (por ejemplo `2023-01-387269`).
  Ordenarlo como texto daría errores cuando los consecutivos tienen distinta cantidad de dígitos,
  así que se construye una llave numérica (`radicado_orden`) con el año, el mes y el consecutivo.
  Esa llave se usa después para identificar el reporte más reciente.
- Se quitan espacios sobrantes en `nit` y `concepto`. La normalización completa del NIT (dígito
  de verificación, tipo de dato) se hace en la etapa de integración.

In [11]:
balances["nit"] = balances["nit"].str.strip()
balances["concepto"] = balances["concepto"].str.strip()

valor_texto = balances["valor"].str.strip()
balances["valor"] = pd.to_numeric(valor_texto, errors="coerce")
no_convertidos = balances["valor"].isna() & valor_texto.notna()

balances["fecha_corte"] = pd.to_datetime(balances["fecha_corte"], errors="coerce")
balances["anio"] = balances["fecha_corte"].dt.year.astype("Int64")

partes = balances["numero_radicado"].str.strip().str.extract(r"^(\d{4})-(\d{2})-(\d+)$")
balances["radicado_orden"] = (
    pd.to_numeric(partes[0]).astype("Int64") * 10**12
    + pd.to_numeric(partes[1]).astype("Int64") * 10**10
    + pd.to_numeric(partes[2]).astype("Int64")
)
balances["codigo_instancia"] = pd.to_numeric(balances["codigo_instancia"], errors="coerce")

print("Valores que no se pudieron convertir a número:", int(no_convertidos.sum()))
print("Valores nulos en 'valor':", int(balances["valor"].isna().sum()))
print("Fechas no válidas:", int(balances["fecha_corte"].isna().sum()))
print("Radicados con formato distinto a AAAA-MM-consecutivo:", int(balances["radicado_orden"].isna().sum()))
print("Fechas de corte presentes:", sorted(balances["fecha_corte"].dt.date.astype(str).unique()))
print()
print(balances[["valor", "fecha_corte", "anio", "radicado_orden", "codigo_instancia"]].dtypes)

Valores que no se pudieron convertir a número: 0
Valores nulos en 'valor': 0
Fechas no válidas: 0
Radicados con formato distinto a AAAA-MM-consecutivo: 0


Fechas de corte presentes: ['2019-12-31', '2020-12-31', '2021-12-31', '2022-12-31', '2023-12-31', '2024-12-31', '2025-12-31']

valor                      float64
fecha_corte         datetime64[us]
anio                         Int64
radicado_orden               Int64
codigo_instancia             int64
dtype: object


> **Interpretación:** las 850.357 filas se convirtieron sin pérdidas: ningún `valor` quedó sin convertir ni nulo, todas las fechas son válidas y todos los radicados tienen el formato `AAAA-MM-consecutivo`. Las siete fechas de corte presentes son el 31 de diciembre de 2019 a 2025, como se pidió en la extracción. Los valores son enteros y están en **miles de pesos**. Se verificó con la base 2: el activo total de una misma empresa y año es unas 10⁹ veces mayor aquí que allá, porque la base 2 está en billones de pesos. Esto hay que tenerlo en cuenta en la integración si se comparan montos entre las dos bases.

#### Duplicados exactos

En la extracción se vio que 2022 trae casi el doble de filas que los demás años. Primero se
eliminan las filas **idénticas en todas sus columnas**: son copias del mismo dato y no aportan
información.

In [12]:
filas_antes = balances.groupby("anio").size()
balances = balances.drop_duplicates().reset_index(drop=True)
filas_despues = balances.groupby("anio").size()

trazabilidad.append(("Eliminar duplicados exactos", len(balances),
                     "Filas idénticas en todas las columnas"))

print(pd.DataFrame({"antes": filas_antes, "después": filas_despues,
                    "eliminadas": filas_antes - filas_despues}))

       antes  después  eliminadas
anio                             
2019   93378    93378           0
2020   98023    98013          10
2021  103061   102966          95
2022  212064   108291      103773
2023  114737   114737           0
2024  113098   113098           0
2025  115996   115996           0


> **Interpretación:** se eliminaron **103.878 filas duplicadas exactas**. Casi todas (103.773) son de 2022, que pasa de 212.064 a 108.291 filas y queda en el mismo rango que los demás años (entre 93.000 y 116.000). Esto confirma lo que se vio en la extracción: la fuente publicó dos veces los registros de ese corte. En 2020 y 2021 también había unos pocos duplicados (10 y 95 filas). Los demás años no tenían.

#### Retransmisiones: un solo reporte por empresa y año

Una empresa puede enviar su balance más de una vez para el mismo corte, por ejemplo para corregir
un error. Cada envío tiene su propio `numero_radicado`. Si se dejan todos, la empresa quedaría con
dos valores distintos para la misma cuenta y el mismo año.

**Regla:** por cada empresa (`nit`) y año se conserva solo el reporte con el `numero_radicado`
más reciente, porque es la versión corregida y vigente. Si dos reportes tienen el mismo radicado,
se desempata con el `codigo_instancia` más alto.

Esto también resuelve los casos en que una empresa aparece con dos tipos de estado el mismo año
(por ejemplo *Individual* y *Separado*): se toma el último que reportó.

In [13]:
llave = ["nit", "anio"]
reportes = (balances[llave + ["numero_radicado", "radicado_orden", "codigo_instancia", "punto_entrada"]]
            .drop_duplicates())

reportes_por_empresa = reportes.groupby(llave).size()
print("Empresa-año con más de un reporte:", int((reportes_por_empresa > 1).sum()),
      f"de {len(reportes_por_empresa):,}")
print("Distribución del número de reportes por empresa-año:")
print(reportes_por_empresa.value_counts().sort_index().rename("empresa-año"))

# Ejemplo de una empresa con varios reportes en el mismo año
con_varios = reportes_por_empresa[reportes_por_empresa > 1].index
if len(con_varios):
    ejemplo = con_varios[0]
    print(f"\nEjemplo: NIT {ejemplo[0]}, año {ejemplo[1]}")
    print(reportes.set_index(llave).loc[[ejemplo]].to_string())

# Reporte vigente: el radicado más reciente (y el código de instancia más alto si empatan)
vigentes = (reportes.sort_values(llave + ["radicado_orden", "codigo_instancia"])
                    .drop_duplicates(llave, keep="last")
                    [llave + ["numero_radicado", "codigo_instancia"]])

filas_antes = len(balances)
balances = balances.merge(vigentes, on=llave + ["numero_radicado", "codigo_instancia"], how="inner")
trazabilidad.append(("Conservar el radicado más reciente", len(balances),
                     "Retransmisiones del mismo corte por la misma empresa"))
print(f"\nFilas eliminadas por retransmisión: {filas_antes - len(balances):,}")

# Control: dentro del reporte vigente, cada cuenta debe aparecer una sola vez
repetidas = balances.duplicated(llave + ["concepto"], keep=False)
print("Cuentas repetidas dentro de un mismo reporte:", int(repetidas.sum()))

Empresa-año con más de un reporte: 238 de 75,829
Distribución del número de reportes por empresa-año:
1    75591
2      236
3        2
Name: empresa-año, dtype: int64

Ejemplo: NIT 800011161, año 2021
               numero_radicado    radicado_orden  codigo_instancia                       punto_entrada
nit       anio                                                                                        
800011161 2021  2022-01-313300  2022010000313300            305013  40 NIIF Pymes - Individual Grupo 2
          2021  2023-01-000478  2023010000000478            334934    50 NIIF Pymes - Separado Grupo 2



Filas eliminadas por retransmisión: 2,333
Cuentas repetidas dentro de un mismo reporte: 0


> **Interpretación:** de 75.829 combinaciones empresa-año, **238 (0,3 %) tenían más de un reporte**: 236 con dos y 2 con tres. Al conservar solo el radicado más reciente se eliminaron 2.333 filas. El ejemplo muestra un caso típico: la empresa envió su balance de 2021 en enero de 2022 como estado *individual* y lo volvió a enviar en enero de 2023 como estado *separado*; se conserva el último envío. Después de este paso, **ninguna cuenta aparece repetida dentro de un mismo reporte**, así que cada empresa-año tiene un único valor por cuenta y se puede pivotear sin agregar datos.

#### Homologación de cuentas entre taxonomías

Las empresas reportan con distintas taxonomías (NIIF Plenas o NIIF Pymes, estados individuales o
separados), y una misma cuenta puede tener nombres ligeramente distintos entre ellas. Para
unificarlas se construye un **diccionario de homologación** a partir de `CUENTAS`, la lista de
nombres que se definió en la extracción:

- La **llave** es el nombre del concepto normalizado: minúsculas, sin tildes y sin espacios
  repetidos. Así no fallan diferencias de forma como mayúsculas o tildes.
- El **valor** es el nombre estándar de la cuenta (`activo_total`, `pasivo_corriente`, etc.).

Del `punto_entrada` se derivan además dos columnas descriptivas que pueden servir más adelante:
el **marco normativo** (NIIF Plenas o NIIF Pymes) y el **tipo de estado** (Individual o
Separado).

In [14]:
import unicodedata

import numpy as np

def normalizar_texto(texto):
    """Minúsculas, sin tildes y con un solo espacio entre palabras."""
    sin_tildes = unicodedata.normalize("NFKD", str(texto)).encode("ascii", "ignore").decode()
    return " ".join(sin_tildes.lower().split())

HOMOLOGACION = {normalizar_texto(nombre): cuenta
                for cuenta, nombres in CUENTAS.items() for nombre in nombres}

conceptos_unicos = balances["concepto"].dropna().unique()
mapa = {c: HOMOLOGACION.get(normalizar_texto(c)) for c in conceptos_unicos}
balances["cuenta"] = balances["concepto"].map(mapa)

print("Diccionario de homologación (concepto en la fuente -> cuenta estándar):")
for concepto, cuenta in sorted(mapa.items(), key=lambda x: str(x[1])):
    print(f"  {concepto:70s} -> {cuenta}")
print("\nFilas sin cuenta homologada:", int(balances["cuenta"].isna().sum()))

balances["marco_normativo"] = np.where(balances["punto_entrada"].str.contains("Plenas"),
                                       "NIIF Plenas", "NIIF Pymes")
balances["tipo_estado"] = np.where(balances["punto_entrada"].str.contains("Separado"),
                                   "Separado", "Individual")
print("\nFilas por taxonomía después de homologar:")
print(balances.groupby(["marco_normativo", "tipo_estado", "punto_entrada"]).size().rename("filas"))

Diccionario de homologación (concepto en la fuente -> cuenta estándar):
  Activos corrientes totales                                             -> activo_corriente
  Total de activos                                                       -> activo_total
  Capital emitido                                                        -> capital_emitido
  Cuentas comerciales por cobrar y otras cuentas por cobrar corrientes   -> deudores_comerciales
  Efectivo y equivalentes al efectivo                                    -> efectivo
  Ganancias acumuladas                                                   -> ganancias_acumuladas
  Inventarios corrientes                                                 -> inventarios
  Pasivos corrientes totales                                             -> pasivo_corriente
  Total pasivos                                                          -> pasivo_total
  Patrimonio total                                                       -> patrimonio_total

Filas sin c


Filas por taxonomía después de homologar:
marco_normativo  tipo_estado  punto_entrada                      
NIIF Plenas      Individual   10 NIIF Plenas - Individual Grupo 1    111567
                              10 NIIF Plenas - Individuales           36595
                 Separado     20 NIIF Plenas - Separado Grupo 1       43960
                              20 NIIF Plenas - Separados              15281
NIIF Pymes       Individual   40 NIIF Pymes - Individual Grupo 2     354418
                              40 NIIF Pymes - Individuales           124306
                 Separado     50 NIIF Pymes - Separado Grupo 2        43153
                              50 NIIF Pymes - Separados               14866
Name: filas, dtype: int64


> **Interpretación:** los 10 nombres de concepto presentes en los datos se homologaron a una cuenta estándar y **ninguna fila quedó sin cuenta**. Entre 2019 y 2025, en las ocho variantes de punto de entrada, la fuente usa el mismo nombre para cada cuenta, así que el diccionario resultó uno a uno. Aun así, la normalización (sin tildes ni diferencias de mayúsculas) deja el proceso preparado si en otro corte aparece una variante. Por marco normativo, el 64,3 % de las filas son de NIIF Pymes individual, el 19,9 % de NIIF Plenas individual, el 8,0 % de NIIF Plenas separado y el 7,8 % de NIIF Pymes separado.

#### Pivoteo: de formato largo a ancho

Ahora cada fila es una empresa, un año y una cuenta. Se pivotea para que quede **una fila por
empresa y año** y **una columna por cuenta**. Además se conservan los datos del reporte de
origen (fecha de corte, radicado, punto de entrada, marco normativo y tipo de estado) para poder
rastrear cada fila hasta la fuente.

Si una empresa no reportó alguna cuenta, esa celda queda **vacía (nulo)**. No se rellena con
cero en esta etapa: que una cuenta no aparezca no significa necesariamente que valga cero. Esa
decisión es una regla de negocio y se documenta en la etapa de indicadores.

In [15]:
ancho = balances.pivot(index=llave, columns="cuenta", values="valor").reset_index()
ancho.columns.name = None

metadatos = (balances[llave + ["fecha_corte", "numero_radicado", "punto_entrada",
                               "marco_normativo", "tipo_estado"]]
             .drop_duplicates(llave))
balances_limpios = metadatos.merge(ancho, on=llave, how="inner")

orden_cuentas = [c for c in CUENTAS if c in balances_limpios.columns]
balances_limpios = (balances_limpios[llave + ["fecha_corte", "numero_radicado", "punto_entrada",
                                             "marco_normativo", "tipo_estado"] + orden_cuentas]
                    .sort_values(llave).reset_index(drop=True))
balances_limpios["anio"] = balances_limpios["anio"].astype(int)

assert not balances_limpios.duplicated(llave).any(), "Hay empresas repetidas en el mismo año"

print("Dimensiones de la tabla ancha:", balances_limpios.shape)
print("Empresas distintas:", balances_limpios["nit"].nunique())
print("\nEmpresas por año:")
print(balances_limpios.groupby("anio").size().rename("empresas"))
print("\nNulos por cuenta (empresa-año sin ese dato):")
nulos = balances_limpios[orden_cuentas].isna().sum()
print(pd.DataFrame({"nulos": nulos, "%": (nulos / len(balances_limpios) * 100).round(2)}))
balances_limpios.head()

Dimensiones de la tabla ancha: (75829, 17)
Empresas distintas: 12761

Empresas por año:
anio
2019     9495
2020     9950
2021    10435
2022    10990
2023    11647
2024    11490
2025    11822
Name: empresas, dtype: int64

Nulos por cuenta (empresa-año sin ese dato):
                      nulos      %
activo_corriente          0   0.00
activo_total              0   0.00
pasivo_corriente         65   0.09
pasivo_total              0   0.00
patrimonio_total          0   0.00
efectivo                  0   0.00
inventarios           13343  17.60
deudores_comerciales    717   0.95
capital_emitido           4   0.01
ganancias_acumuladas     15   0.02


,nit,anio,fecha_corte,numero_radicado,punto_entrada,marco_normativo,tipo_estado,activo_corriente,activo_total,pasivo_corriente,pasivo_total,patrimonio_total,efectivo,inventarios,deudores_comerciales,capital_emitido,ganancias_acumuladas
0,800000276,2019,2019-12-31,2020-01-147029,40 NIIF Pymes - Individuales,NIIF Pymes,Individual,62655234.0,184089776.0,51170758.0,118087246.0,66002530.0,1430193.0,19954573.0,24958457.0,3500000.0,-671303.0
1,800000276,2020,2020-12-31,2021-01-265062,40 NIIF Pymes - Individuales,NIIF Pymes,Individual,70960765.0,192839599.0,58680073.0,128343041.0,64496558.0,1569192.0,20792400.0,30889109.0,3500000.0,184586.0
2,800000276,2021,2021-12-31,2022-01-317544,40 NIIF Pymes - Individual Grupo 2,NIIF Pymes,Individual,88837394.0,218942396.0,81230425.0,138622880.0,80319516.0,1326682.0,20473334.0,41456884.0,3576000.0,14089906.0
3,800000276,2022,2022-12-31,2023-01-366968,40 NIIF Pymes - Individual Grupo 2,NIIF Pymes,Individual,145169269.0,278713426.0,123610309.0,183621483.0,95091943.0,2331706.0,42997280.0,60436680.0,3576000.0,31430109.0
4,800000276,2023,2023-12-31,2024-01-248290,40 NIIF Pymes - Individual Grupo 2,NIIF Pymes,Individual,122927377.0,269282309.0,101370290.0,169549077.0,99733232.0,4200198.0,33980237.0,54444946.0,3576000.0,36356646.0


> **Interpretación:** la tabla final tiene **75.829 filas (una por empresa y año) de 12.761 empresas** y 17 columnas: la llave (`nit`, `anio`), cinco columnas del reporte de origen y las diez cuentas. El número de empresas por año coincide con el de la extracción (de 9.495 en 2019 a 11.822 en 2025), lo que confirma que la limpieza no perdió empresas. Activo corriente, activo total, pasivo total, patrimonio total y efectivo están completos. **Inventarios** falta en el 17,6 % de los casos, sobre todo en empresas de servicios que no lo reportan. En menor medida faltan deudores comerciales (0,95 %), pasivo corriente (65 casos), ganancias acumuladas (15) y capital emitido (4). Los nulos de pasivo corriente son importantes porque impiden calcular la liquidez. Cómo tratarlos es una regla de negocio que se define en la etapa de indicadores.

#### Resumen de la limpieza y guardado

Se guardan dos archivos en `data/processed/`:

| Archivo | Contenido |
|---|---|
| `balances_limpios.csv` | Una fila por empresa y año con las 10 cuentas en columnas numéricas |
| `trazabilidad_limpieza.csv` | Filas que quedan después de cada paso y el motivo de lo eliminado |

La trazabilidad sirve de insumo para la tabla de calidad de datos de la etapa de integración.

In [16]:
tabla_traza = pd.DataFrame(trazabilidad, columns=["paso", "filas_resultantes", "motivo"])
tabla_traza["filas_eliminadas"] = (-tabla_traza["filas_resultantes"].diff()).fillna(0).astype(int)
tabla_traza = pd.concat([tabla_traza, pd.DataFrame([{
    "paso": "Pivoteo a formato ancho (empresa-año)",
    "filas_resultantes": len(balances_limpios),
    "motivo": "Cambio de forma: una fila por empresa y año",
    "filas_eliminadas": 0,
}])], ignore_index=True)[["paso", "filas_resultantes", "filas_eliminadas", "motivo"]]

balances_limpios.to_csv(DATA_PROCESSED / "balances_limpios.csv", index=False)
tabla_traza.to_csv(DATA_PROCESSED / "trazabilidad_limpieza.csv", index=False)

print("Archivos guardados en data/processed/: balances_limpios.csv, trazabilidad_limpieza.csv")
tabla_traza

Archivos guardados en data/processed/: balances_limpios.csv, trazabilidad_limpieza.csv


,paso,filas_resultantes,filas_eliminadas,motivo
0,Carga de balances_raw.csv,850357,0,Archivo generado en la extracción
1,Eliminar duplicados exactos,746479,103878,Filas idénticas en todas las columnas
2,Conservar el radicado más reciente,744146,2333,Retransmisiones del mismo corte por la misma e...
3,Pivoteo a formato ancho (empresa-año),75829,0,Cambio de forma: una fila por empresa y año


> **Interpretación:** de 850.357 filas descargadas quedaron 744.146 válidas. Se eliminaron 103.878 por ser duplicados exactos (12,2 %) y 2.333 por ser retransmisiones (0,3 %). Al pivotear quedaron **75.829 registros empresa-año**, uno por cada balance de cierre vigente.

**Archivos generados en `data/processed/`:**

| Archivo | Filas | Contenido |
|---|---|---|
| `balances_limpios.csv` | 75.829 | Una fila por NIT y año, con las 10 cuentas en miles de pesos |
| `trazabilidad_limpieza.csv` | 4 | Pasos de la limpieza y filas eliminadas en cada uno |

*Siguiente paso:* normalizar el NIT, cruzar con la base de las 10.000 empresas y validar la ecuación contable.

### 1.3 Integración con las 10.000 empresas y validación de calidad

La tabla de balances limpios tiene una fila por empresa y año. En esta sección se le agregan
los datos de sector, ubicación, ingresos y ganancia del ranking del mismo año. Se conserva el
histórico completo de balances y se mide qué tan bien cruzan las dos fuentes.

#### Carga de los insumos

Se usan los balances y la trazabilidad que dejó la sección 1.2, junto con la base de las 10.000
empresas de la sección 1.1. Esta sección se puede ejecutar por separado desde la raíz del
proyecto o desde `notebooks/`. Si falta el ranking, basta ejecutar en la sección 1.1 la
configuración, la función paginada y la descarga de la base 2.

In [1]:
from pathlib import Path

import pandas as pd

raiz_c3 = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
procesados_c3 = raiz_c3 / "data" / "processed"
ruta_ranking = raiz_c3 / "data" / "raw" / "empresas_10000_raw.csv"

if not ruta_ranking.is_file():
    raise FileNotFoundError("Falta empresas_10000_raw.csv. Ejecuta la descarga de la base 2 en la sección 1.1.")

balances_c3 = pd.read_csv(
    procesados_c3 / "balances_limpios.csv", dtype={"nit": "string", "anio": "string"}
)
ranking_c3 = pd.read_csv(ruta_ranking, dtype="string")
traza_c2 = pd.read_csv(procesados_c3 / "trazabilidad_limpieza.csv")

print(f"Balances de C2: {len(balances_c3):,} filas")
print(f"Ranking: {len(ranking_c3):,} filas")
print(f"Pasos registrados en la trazabilidad de C2: {len(traza_c2)}")

Balances de C2: 75,829 filas
Ranking: 50,000 filas
Pasos registrados en la trazabilidad de C2: 4


> **Interpretación:** se parte de 75.829 balances y 50.000 registros del ranking. La
> trazabilidad de C2 permite explicar qué se eliminó durante la limpieza.

#### Normalización de llaves y montos

El NIT se lee como texto. Se quitan espacios externos y el sufijo decimal de la API
(`.00`), o el dígito de verificación si viene separado por un guion. Se esperan nueve dígitos
en el NIT base. El año se convierte a un entero de cuatro dígitos. Se comprueba la llave
`(nit, anio)` antes de cruzar. Se contabilizan las filas del ranking con llaves inválidas y se
consolidan las que quedan idénticas en los campos usados. Si quedan valores distintos para la
misma llave, la ejecución se detiene con el conteo para revisar la fuente.

Santiago verificó que los balances están en **miles de pesos colombianos** y que el ranking
publica sus cifras en **billones de pesos**. Se retira el símbolo de moneda, se conserva el
signo y se multiplica por **10⁹** para guardar ingresos y ganancia en miles de pesos. Las
celdas vacías y los montos que no se puedan convertir quedan como nulos, con un conteo de los
errores de conversión. [Ficha del ranking](https://www.datos.gov.co/d/6cat-2gcs).

In [2]:
def normalizar_nit_c3(serie):
    """Extrae el NIT base de nueve dígitos en los formatos esperados."""
    return serie.str.strip().str.extract(r"^([0-9]{9})(?:\.0+|\s*-\s*[0-9])?$", expand=False)


def normalizar_anio_c3(serie):
    """Acepta años como 2025 o 2025.00 y conserva los inválidos como nulos."""
    anio = serie.str.strip().str.extract(r"^([1-9][0-9]{3})(?:\.0+)?$", expand=False)
    return anio.astype("Int64")


llave_c3 = ["nit", "anio"]
balances_c3["nit"] = normalizar_nit_c3(balances_c3["nit"])
balances_c3["anio"] = normalizar_anio_c3(balances_c3["anio"])
ranking_c3["nit"] = normalizar_nit_c3(ranking_c3["nit"])
ranking_c3["anio"] = normalizar_anio_c3(ranking_c3["a_o_de_corte"])

llaves_invalidas = {
    "Balances": int(balances_c3[llave_c3].isna().any(axis=1).sum()),
    "Ranking": int(ranking_c3[llave_c3].isna().any(axis=1).sum()),
}
duplicados_balances = int(balances_c3.duplicated(llave_c3, keep=False).sum())
if llaves_invalidas["Balances"] or duplicados_balances:
    raise ValueError(
        f"Revisar la llave de balances de C2: {llaves_invalidas['Balances']} filas con llave "
        f"inválida y {duplicados_balances} en llaves repetidas."
    )

campos_ranking = {
    "macrosector": "macrosector",
    "regi_n": "region",
    "departamento_domicilio": "departamento",
    "ciudad_domicilio": "ciudad",
    "ingresos_operacionales": "ingresos_operacionales_miles_cop",
    "ganancia_p_rdida": "ganancia_perdida_miles_cop",
}
cortes_ranking = ranking_c3["anio"].dropna().unique()
ranking_integrable = (
    ranking_c3[llave_c3 + list(campos_ranking)]
    .dropna(subset=llave_c3)
    .rename(columns=campos_ranking)
)
for campo in ("macrosector", "region", "departamento", "ciudad"):
    ranking_integrable[campo] = ranking_integrable[campo].str.strip().replace("", pd.NA)

factor_billones_a_miles = 10**9
montos_invalidos = {}
for campo in ("ingresos_operacionales_miles_cop", "ganancia_perdida_miles_cop"):
    texto = ranking_integrable[campo].str.strip().replace("", pd.NA)
    monto = texto.str.extract(r"^([+-]?)\s*\$?\s*([0-9]+(?:\.[0-9]+)?)$")
    billones = pd.to_numeric(monto[0] + monto[1], errors="coerce")
    montos_invalidos[campo] = int((texto.notna() & billones.isna()).sum())
    ranking_integrable[campo] = (billones * factor_billones_a_miles).round().astype("Int64")

filas_ranking_validas = len(ranking_integrable)
ranking_integrable = ranking_integrable.drop_duplicates()
duplicados_ranking_eliminados = filas_ranking_validas - len(ranking_integrable)
conflictos_ranking = ranking_integrable.duplicated(llave_c3, keep=False)
if conflictos_ranking.any():
    ejemplos = ranking_integrable.loc[conflictos_ranking, llave_c3].drop_duplicates().head()
    raise ValueError(
        f"El ranking tiene {int(conflictos_ranking.sum())} filas con valores distintos "
        f"para la misma llave. Primeras llaves:\n{ejemplos.to_string(index=False)}"
    )

print("Filas con llave inválida:", llaves_invalidas)
print("Filas de balances en llaves repetidas:", duplicados_balances)
print("Duplicados equivalentes retirados del ranking:", duplicados_ranking_eliminados)
print("Montos del ranking que no se pudieron convertir:", montos_invalidos)

Filas con llave inválida: {'Balances': 0, 'Ranking': 0}
Filas de balances en llaves repetidas: 0
Duplicados equivalentes retirados del ranking: 0
Montos del ranking que no se pudieron convertir: {'ingresos_operacionales_miles_cop': 0, 'ganancia_perdida_miles_cop': 0}


> **Interpretación:** ambas fuentes quedaron con una llave única por NIT y año. En esta
> extracción no hubo llaves inválidas, duplicados ni errores de conversión de montos.

#### Cruce y cobertura por año

Se hace una unión izquierda desde los balances por NIT y año, con validación uno a uno. La
columna `estado_cruce` indica si hubo coincidencia, si el corte existe en el ranking pero el
NIT quedó sin coincidencia, o si el ranking no tiene ese corte. Usar el mismo año evita
asignar atributos de años posteriores a los balances históricos.

In [3]:
balances_integrados = balances_c3.merge(
    ranking_integrable, on=llave_c3, how="left", validate="one_to_one", indicator=True
)
balances_integrados["estado_cruce"] = (
    balances_integrados.pop("_merge").map({"both": "coincide", "left_only": "sin_coincidencia"})
    .astype("string")
)
balances_integrados.loc[
    ~balances_integrados["anio"].isin(cortes_ranking), "estado_cruce"
] = "sin_corte_ranking"

if len(balances_integrados) != len(balances_c3) or balances_integrados.duplicated(llave_c3).any():
    raise ValueError("El cruce debe conservar el número de balances y su llave única.")

estados_cruce = ["coincide", "sin_coincidencia", "sin_corte_ranking"]
cobertura_c3 = (
    pd.crosstab(balances_integrados["anio"], balances_integrados["estado_cruce"])
    .reindex(columns=estados_cruce, fill_value=0)
    .rename_axis(columns=None)
)
cobertura_c3["total"] = cobertura_c3.sum(axis=1)
cobertura_c3["porcentaje_coincide"] = (
    cobertura_c3["coincide"].div(cobertura_c3["total"]).mul(100).round(2)
    .where(cobertura_c3.index.isin(cortes_ranking))
)
cobertura_c3

,coincide,sin_coincidencia,sin_corte_ranking,total,porcentaje_coincide
anio,,,,,
2019,0,0,9495,9495,NaN
2020,0,0,9950,9950,NaN
2021,8817,1618,0,10435,84.49
2022,8921,2069,0,10990,81.17
2023,8464,3183,0,11647,72.67
2024,8493,2997,0,11490,73.92
2025,8821,3001,0,11822,74.62


> **Interpretación:** se conservaron los 75.829 balances. En los cortes cubiertos por el
> ranking cruzaron 43.516 de 56.384 balances (77,18 %). La cobertura fue mayor en 2021
> (84,49 %) y menor en 2023 (72,67 %). Los 19.445 balances de 2019 y 2020 se conservaron con
> los campos del ranking vacíos porque la fuente no tiene esos cortes.

#### Validación de la ecuación contable

Se calcula la diferencia entre activo total y la suma de pasivo total y patrimonio total, con
las tres cuentas de C2 en miles de pesos. La diferencia en valor absoluto puede ser de hasta
el **1 % del activo total en valor absoluto**. Con activo cero se exige diferencia cero y el error relativo queda vacío. Si
falta alguna de las tres cuentas, el balance se marca como `no_evaluable`. Se conserva cada
balance con el resultado de la validación.

In [4]:
activo = balances_integrados["activo_total"]
pasivo = balances_integrados["pasivo_total"]
patrimonio = balances_integrados["patrimonio_total"]

tolerancia_contable = 0.01
diferencia = activo - (pasivo + patrimonio)
base_error = activo.abs()
balances_integrados["diferencia_contable_miles_cop"] = diferencia
balances_integrados["error_relativo_contable"] = diferencia.abs().div(base_error.where(base_error.ne(0)))

estados_contables = ["cuadra", "no_cuadra", "no_evaluable"]
estado_contable = pd.Series("no_evaluable", index=balances_integrados.index, dtype="string")
evaluables = diferencia.notna()
dentro_tolerancia = diferencia.abs().le(base_error * tolerancia_contable)
estado_contable.loc[evaluables & dentro_tolerancia] = "cuadra"
estado_contable.loc[evaluables & ~dentro_tolerancia] = "no_cuadra"
balances_integrados["estado_ecuacion_contable"] = estado_contable

conteo_ecuacion = estado_contable.value_counts().reindex(estados_contables, fill_value=0)
conteo_ecuacion.rename("balances").to_frame()

,balances
cuadra,75829
no_cuadra,0
no_evaluable,0


> **Interpretación:** los 75.829 balances satisfacen la ecuación de forma exacta. En este
> archivo no se encontraron diferencias contables ni cuentas faltantes que impidan evaluarla.

#### Resumen de calidad y archivos para la siguiente etapa

Se documentan la cobertura por año, el resultado de la ecuación, los nulos por columna y los
controles de llaves y montos. El resumen conserva la trazabilidad de C2 en filas de cuenta
hasta el pivoteo. A partir de ese paso la unidad es el balance por empresa y año. Se mantienen
los nulos de las cuentas: su tratamiento para calcular indicadores corresponde a la sección
siguiente.

In [5]:
total_balances_c3 = len(balances_integrados)
total_ranking_c3 = len(ranking_c3)

resumen_nulos = balances_integrados.isna().sum().rename("nulos").to_frame()
resumen_nulos["porcentaje"] = resumen_nulos["nulos"].div(total_balances_c3).mul(100).round(2)
nulos_calidad = resumen_nulos.reset_index(names="detalle").assign(
    etapa="C3", control="Nulos por columna", unidad="empresa-año",
    total_referencia=total_balances_c3,
).rename(columns={"nulos": "cantidad"})

ecuacion_calidad = conteo_ecuacion.rename_axis("detalle").rename("cantidad").reset_index().assign(
    etapa="C3", control="Ecuación contable", unidad="empresa-año",
    total_referencia=total_balances_c3,
)

cobertura_calidad = cobertura_c3.reset_index().melt(
    id_vars=["anio", "total"], value_vars=estados_cruce, var_name="detalle", value_name="cantidad"
).assign(etapa="C3", control="Cobertura del cruce", unidad="empresa-año").rename(
    columns={"total": "total_referencia"}
)

controles_calidad = pd.DataFrame(
    [
        ("Llave inválida", "Balances", "empresa-año", llaves_invalidas["Balances"], len(balances_c3)),
        ("Llave repetida", "Balances", "empresa-año", duplicados_balances, len(balances_c3)),
        ("Llave inválida", "Ranking", "fila del ranking", llaves_invalidas["Ranking"], total_ranking_c3),
        ("Llave en conflicto", "Ranking", "fila del ranking", int(conflictos_ranking.sum()), len(ranking_integrable)),
        ("Filas retiradas", "Ranking: llave inválida", "fila del ranking", llaves_invalidas["Ranking"], total_ranking_c3),
        ("Filas retiradas", "Ranking: duplicados equivalentes", "fila del ranking", duplicados_ranking_eliminados, filas_ranking_validas),
        ("Filas retiradas", "Balances durante el cruce", "empresa-año", 0, len(balances_c3)),
        *[
            ("Monto inválido", campo, "fila del ranking", cantidad, filas_ranking_validas)
            for campo, cantidad in montos_invalidos.items()
        ],
    ],
    columns=["control", "detalle", "unidad", "cantidad", "total_referencia"],
).assign(etapa="C3")

traza_calidad = traza_c2.rename(
    columns={"paso": "detalle", "filas_resultantes": "cantidad"}
).assign(etapa="C2", control="Filas resultantes", unidad="fila de cuenta")
paso_pivoteo = traza_calidad["detalle"].str.startswith("Pivoteo")
traza_calidad.loc[paso_pivoteo, "unidad"] = "empresa-año"
retiros_c2 = traza_calidad.loc[
    traza_calidad["filas_eliminadas"].gt(0), ["etapa", "motivo", "unidad", "filas_eliminadas"]
].rename(columns={"motivo": "detalle", "filas_eliminadas": "cantidad"}).assign(
    control="Filas retiradas"
)

resumen_calidad = pd.concat(
    [traza_calidad, retiros_c2, controles_calidad, cobertura_calidad, ecuacion_calidad, nulos_calidad],
    ignore_index=True,
)[["etapa", "control", "detalle", "unidad", "anio", "cantidad", "total_referencia"]]
for campo in ("anio", "total_referencia"):
    resumen_calidad[campo] = resumen_calidad[campo].astype("Int64")
resumen_calidad["porcentaje"] = (
    resumen_calidad["cantidad"]
    .div(resumen_calidad["total_referencia"].where(resumen_calidad["total_referencia"].gt(0)))
    .mul(100)
    .round(2)
)

balances_integrados.to_csv(procesados_c3 / "balances_integrados.csv", index=False)
resumen_calidad.to_csv(procesados_c3 / "resumen_calidad.csv", index=False)

print("Archivos guardados en data/processed/: balances_integrados.csv, resumen_calidad.csv")
resumen_nulos.loc[resumen_nulos["nulos"].gt(0)]

Archivos guardados en data/processed/: balances_integrados.csv, resumen_calidad.csv


,nulos,porcentaje
pasivo_corriente,65,0.09
inventarios,13343,17.60
deudores_comerciales,717,0.95
capital_emitido,4,0.01
ganancias_acumuladas,15,0.02
macrosector,32313,42.61
region,32313,42.61
departamento,32313,42.61
ciudad,32313,42.61
ingresos_operacionales_miles_cop,32313,42.61


> **Interpretación:** inventarios sigue siendo la cuenta con más faltantes (13.343 balances,
> 17,60 %) y hay 65 balances sin pasivo corriente. Los campos del ranking quedan vacíos en los
> 32.313 balances sin coincidencia. Los 103.878 duplicados exactos y las 2.333 filas de
> retransmisiones corresponden a C2. En C3 se conservaron todos los balances.

| Archivo | Contenido |
|---|---|
| `data/processed/balances_integrados.csv` | Una fila por empresa y año, con las cuentas de C2, los campos del ranking y las marcas de calidad. Los montos están en miles de pesos colombianos. |
| `data/processed/resumen_calidad.csv` | Trazabilidad de C2, controles de C3, cobertura por corte, resultados de la ecuación y nulos. |

*Siguiente paso:* calcular los indicadores y construir el modelo estrella a partir de
`balances_integrados.csv`.